In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from ultralytics import YOLO
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

import csv
import os
from datetime import datetime

# ==============================
# Load Models
# ==============================

face_model = YOLO(
    r"E:\mohammad\Programming\Pytthon_4_AI\NTI_CV\face_det\Face detection_enhanced\train\weights\best.pt"
)

emotion_model = load_model(
    "best_emotion_model_scratch_enhanced.keras",
    custom_objects={"preprocess_input": preprocess_input},
)

# ==============================
# Emotion Labels
# ==============================

emotions = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise",
]

# ==============================
# CSV Setup
# ==============================

csv_file = "emotion_log.csv"

if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(
            ["Timestamp", "Person", "Emotion", "Confidence"]
        )

# ==============================
# Camera
# ==============================

cap = cv2.VideoCapture(0)

last_save_time = 0

# ==============================
# Main Loop
# ==============================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Face Detection
    results = face_model(frame)

    current_time = cv2.getTickCount() / cv2.getTickFrequency()

    for person_id, box in enumerate(results[0].boxes.xyxy, start=1):

        x1, y1, x2, y2 = map(int, box)

        person_name = f"Person{person_id}"

        # Crop Face
        face = frame[y1:y2, x1:x2]

        if face.size == 0:
            continue

        # ==============================
        # Preprocess Face
        # ==============================

        face_input = cv2.resize(face, (48, 48))
        face_input = face_input.astype("float32") / 255.0
        face_input = np.reshape(face_input, (1, 48, 48, 3))

        # ==============================
        # Emotion Prediction
        # ==============================

        prediction = emotion_model.predict(face_input, verbose=0)

        emotion_label = np.argmax(prediction)
        confidence = float(np.max(prediction))

        emotion_name = emotions[emotion_label]

        emotion_text = f"{emotion_name} ({confidence:.2f})"

        # ==============================
        # Save to CSV Every 1 Second
        # ==============================

        if current_time - last_save_time >= 1:

            timestamp = datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )

            with open(csv_file, mode="a", newline="") as file:
                writer = csv.writer(file)

                writer.writerow(
                    [
                        timestamp,
                        person_name,
                        emotion_name,
                        round(confidence, 4),
                    ]
                )

            last_save_time = current_time

        # ==============================
        # Draw Bounding Box
        # ==============================

        y_text = y1 - 10 if y1 - 10 > 10 else y1 + 20

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2,
        )

        (text_width, text_height), _ = cv2.getTextSize(
            f"{person_name}: {emotion_text}",
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            2,
        )

        cv2.rectangle(
            frame,
            (x1, y_text - text_height - 5),
            (x1 + text_width, y_text + 5),
            (0, 255, 0),
            -1,
        )

        cv2.putText(
            frame,
            f"{person_name}: {emotion_text}",
            (x1, y_text),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 0),
            2,
        )

    # ==============================
    # Display Frame
    # ==============================

    cv2.imshow("Live Face & Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# ==============================
# Cleanup
# ==============================

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 Face, 126.4ms
Speed: 4.1ms preprocess, 126.4ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 92.0ms
Speed: 2.0ms preprocess, 92.0ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 98.6ms
Speed: 3.1ms preprocess, 98.6ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 139.6ms
Speed: 2.4ms preprocess, 139.6ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 98.3ms
Speed: 2.8ms preprocess, 98.3ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 95.2ms
Speed: 2.2ms preprocess, 95.2ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 130.6ms
Speed: 3.0ms preprocess, 130.6ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Face, 98.6ms
Speed: 2.1ms preprocess, 98.6ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0

KeyboardInterrupt: 

: 